In [16]:
import sys, numpy as np, pandas as pd, sklearn, torch
print("Python :", sys.version.split()[0])
print("numpy  :", np.__version__)
print("pandas :", pd.__version__)
print("sklearn:", sklearn.__version__)
print("torch  :", torch.__version__)

Python : 3.14.6
numpy  : 2.5.0
pandas : 3.0.3
sklearn: 1.9.0
torch  : 2.12.1


In [17]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
N = 8000
# order must match the score-stack order in the outcome cell
OUTCOME_CLASSES = ["SUCCESS", "FAIL_MOP", "FAIL_BANK_VALIDATION", "FAIL_NAME_MISMATCH", "FAIL_AMOUNT_LIMIT"]
print(f"{N} rows, {len(OUTCOME_CLASSES)} classes")

8000 rows, 5 classes


In [18]:
# Payout context
destination_region = rng.choice(["NAM","EU","LATAM","SSA","MENA","SEA","SA"], N, p=[.22,.20,.16,.10,.08,.14,.10])
payout_method      = rng.choice(["dlb_bank","wise","thunes_wallet","payoneer","paypal","ach","wire"], N, p=[.30,.18,.12,.16,.10,.08,.06])
amount_usd         = (rng.gamma(2.0, 160, N) + 20).round(2)
is_batch_payout    = rng.binomial(1, 0.60, N)

# Account history
account_age_days         = rng.integers(1, 2400, N)
prior_successful_payouts = rng.poisson(account_age_days / 90.0)
historical_failure_rate  = np.clip(rng.beta(1.5, 12, N) + (account_age_days < 120)*0.1, 0, 1).round(3)
days_since_last_payout   = rng.integers(0, 400, N)
top_rated_status         = np.clip((prior_successful_payouts > 15).astype(int) + (prior_successful_payouts > 40).astype(int), 0, 3)

# Method & verification
mop_age_days            = rng.integers(0, 1500, N)
mop_verified            = (rng.uniform(0, 1, N) < 0.88).astype(int)
bank_detail_age_days    = rng.integers(0, 2200, N)
p_invalid               = 0.03 + 0.07*(bank_detail_age_days > 730) + 0.04*(bank_detail_age_days > 1460) + 0.02*is_batch_payout
bank_account_valid      = (rng.uniform(0, 1, N) > p_invalid).astype(int)
name_has_special_chars  = rng.binomial(1, 0.14, N)
name_match_score        = np.clip(rng.beta(9, 1.2, N) - name_has_special_chars*rng.uniform(0.15, 0.5, N), 0, 1).round(3)
recent_bank_change_flag = rng.binomial(1, 0.12, N)
print("16 features generated.")

16 features generated.


In [20]:
amount_z = (amount_usd - amount_usd.mean()) / amount_usd.std()

s_success = 2.6 + 0.04*np.minimum(prior_successful_payouts, 50) + 0.5*top_rated_status \
            - 3.0*historical_failure_rate + 0.0004*account_age_days
s_mop     = 3.0*(1-mop_verified) + 1.7*(mop_age_days < 30) + 1.6*is_batch_payout \
            + 1.0*recent_bank_change_flag + 0.6*(days_since_last_payout > 180)   # recent_bank_change now MOP-only
s_bank    = 2.6*(1-bank_account_valid) + 1.3*(bank_detail_age_days > 730) + 0.9*is_batch_payout \
            + 0.5*(payout_method == "dlb_bank")
s_name    = 2.9*(name_match_score < 0.6) + 2.2*name_has_special_chars + 1.0*(name_match_score < 0.8)
s_amount  = 2.4*np.clip(amount_z, 0, None) + 3.2*(amount_usd > 900) + 2.0*(amount_usd > 1500)

scores  = np.vstack([s_success, s_mop, s_bank, s_name, s_amount]).T + rng.gumbel(0, 0.4, (N, 5))
outcome = np.array(OUTCOME_CLASSES)[scores.argmax(axis=1)]
print("scores shape:", scores.shape)   # (8000, 5)

scores shape: (8000, 5)


In [21]:
df = pd.DataFrame({
    "amount_usd": amount_usd, "payout_method": payout_method, "destination_region": destination_region,
    "is_batch_payout": is_batch_payout, "account_age_days": account_age_days,
    "prior_successful_payouts": prior_successful_payouts, "historical_failure_rate": historical_failure_rate,
    "days_since_last_payout": days_since_last_payout, "top_rated_status": top_rated_status,
    "mop_age_days": mop_age_days, "mop_verified": mop_verified, "recent_bank_change_flag": recent_bank_change_flag,
    "bank_account_valid": bank_account_valid, "bank_detail_age_days": bank_detail_age_days,
    "name_match_score": name_match_score, "name_has_special_chars": name_has_special_chars,
    "outcome": outcome,
})
df.to_csv("payouts.csv", index=False)
print("Shape:", df.shape, "→ 16 features + outcome")
print(df["outcome"].value_counts(normalize=True).round(3))
print("Missing values:", df.isna().sum().sum())   # expect 0 (no 'NA' code anymore)
df.head()

Shape: (8000, 17) → 16 features + outcome
outcome
SUCCESS                 0.600
FAIL_MOP                0.113
FAIL_NAME_MISMATCH      0.102
FAIL_BANK_VALIDATION    0.101
FAIL_AMOUNT_LIMIT       0.084
Name: proportion, dtype: float64
Missing values: 0


,amount_usd,payout_method,destination_region,is_batch_payout,account_age_days,prior_successful_payouts,historical_failure_rate,days_since_last_payout,top_rated_status,mop_age_days,mop_verified,recent_bank_change_flag,bank_account_valid,bank_detail_age_days,name_match_score,name_has_special_chars,outcome
0,547.87,payoneer,SSA,0,952,10,0.012,142,0,609,1,1,1,1430,0.803,0,SUCCESS
1,605.90,payoneer,SEA,1,2080,23,0.024,6,1,103,1,1,1,663,0.917,0,SUCCESS
2,170.30,ach,SEA,0,1740,21,0.004,303,1,109,1,0,1,19,0.951,0,SUCCESS
3,226.38,paypal,EU,1,1885,25,0.173,334,1,961,1,0,1,788,0.836,0,SUCCESS
4,265.57,payoneer,EU,0,2120,18,0.183,322,1,1057,1,0,1,2155,0.901,0,SUCCESS


In [22]:
print("Missing values:", df.isna().sum().sum())   # should now be 0

Missing values: 0


In [23]:
import pandas as pd

df = pd.read_csv("payouts.csv")     # load what we generated and saved
TARGET = "outcome"
X = df.drop(columns=TARGET)         # X = all 26 features (the inputs)
y = df[TARGET]                      # y = the label we want to predict
print("X:", X.shape, "| classes in y:", y.nunique())

X: (8000, 16) | classes in y: 5


In [24]:
from sklearn.model_selection import train_test_split

# Models need numbers, not text. One-hot encoding makes one 0/1 column per category value
# (e.g. payout_method_wise, payout_method_ach …). We avoid integer-coding categories because
# that would imply a fake order (wise=1, ach=2 doesn't mean ach > wise).
X_encoded = pd.get_dummies(X, columns=["payout_method", "destination_region"])
print("After one-hot:", X_encoded.shape[1], "columns")

# Hold out 20% the model never trains on, so our score reflects generalization, not memorization.
# stratify=y keeps each class's proportion the same in train and test — critical with imbalance.
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y)
print("train:", X_train.shape, "| test:", X_test.shape)

After one-hot: 28 columns
train: (6400, 28) | test: (1600, 28)


In [25]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

tree = DecisionTreeClassifier(
    max_depth=6,                 # limit depth so it generalizes instead of memorizing
    class_weight="balanced",     # up-weight rare classes so it doesn't just predict SUCCESS
    random_state=42)
tree.fit(X_train, y_train)       # learn the yes/no split rules from training data
pred = tree.predict(X_test)      # predict on the held-out set

print("Accuracy :", round(accuracy_score(y_test, pred), 3))
print("Macro-F1 :", round(f1_score(y_test, pred, average="macro"), 3))
print()
print(classification_report(y_test, pred))

Accuracy : 0.847
Macro-F1 : 0.795

                      precision    recall  f1-score   support

   FAIL_AMOUNT_LIMIT       0.82      0.90      0.85       134
FAIL_BANK_VALIDATION       0.76      0.54      0.63       162
            FAIL_MOP       0.79      0.57      0.66       181
  FAIL_NAME_MISMATCH       0.92      0.96      0.94       164
             SUCCESS       0.86      0.92      0.89       959

            accuracy                           0.85      1600
           macro avg       0.83      0.78      0.80      1600
        weighted avg       0.84      0.85      0.84      1600



In [26]:
from sklearn.metrics import confusion_matrix

# Which features drove the decisions? Should echo our design: name match, amount, bank valid, mop verified…
imp = sorted(zip(X_encoded.columns, tree.feature_importances_), key=lambda t: -t[1])
print("Top 10 features used:")
for name, score in imp[:10]:
    print(f"  {name:34s} {score:.3f}")

# Confusion matrix: rows = actual, columns = predicted. The diagonal is correct predictions.
labels = sorted(y.unique())
cm = confusion_matrix(y_test, pred, labels=labels)
print("\nConfusion matrix:")
print(pd.DataFrame(cm, index=labels, columns=labels))

Top 10 features used:
  amount_usd                         0.281
  name_match_score                   0.270
  mop_verified                       0.166
  bank_account_valid                 0.149
  prior_successful_payouts           0.064
  name_has_special_chars             0.042
  is_batch_payout                    0.009
  account_age_days                   0.007
  bank_detail_age_days               0.007
  days_since_last_payout             0.002

Confusion matrix:
                      FAIL_AMOUNT_LIMIT  FAIL_BANK_VALIDATION  FAIL_MOP  \
FAIL_AMOUNT_LIMIT                   120                     0         1   
FAIL_BANK_VALIDATION                  2                    87         4   
FAIL_MOP                              3                     5       103   
FAIL_NAME_MISMATCH                    0                     1         1   
SUCCESS                              22                    22        22   

                      FAIL_NAME_MISMATCH  SUCCESS  
FAIL_AMOUNT_LIMIT         

In [27]:
from sklearn.metrics import accuracy_score, f1_score

print(f"{'config':36s}{'accuracy':>10}{'macro-F1':>10}")
for depth in [6, 12, None]:
    for cw in [None, "balanced"]:
        t = DecisionTreeClassifier(max_depth=depth, class_weight=cw, random_state=42).fit(X_train, y_train)
        p = t.predict(X_test)
        cfg = f"depth={depth}, class_weight={cw}"
        print(f"{cfg:36s}{accuracy_score(y_test,p):10.3f}{f1_score(y_test,p,average='macro'):10.3f}")

config                                accuracy  macro-F1
depth=6, class_weight=None               0.838     0.783
depth=6, class_weight=balanced           0.847     0.795
depth=12, class_weight=None              0.829     0.774
depth=12, class_weight=balanced          0.782     0.744
depth=None, class_weight=None            0.794     0.734
depth=None, class_weight=balanced        0.788     0.736


In [28]:
from sklearn.metrics import classification_report

tree = DecisionTreeClassifier(max_depth=12, random_state=42).fit(X_train, y_train)
pred = tree.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 3),
      "| Macro-F1:", round(f1_score(y_test, pred, average="macro"), 3))
print(classification_report(y_test, pred))

Accuracy: 0.829 | Macro-F1: 0.774
                      precision    recall  f1-score   support

   FAIL_AMOUNT_LIMIT       0.86      0.84      0.85       134
FAIL_BANK_VALIDATION       0.68      0.55      0.61       162
            FAIL_MOP       0.65      0.61      0.63       181
  FAIL_NAME_MISMATCH       0.93      0.86      0.90       164
             SUCCESS       0.86      0.91      0.88       959

            accuracy                           0.83      1600
           macro avg       0.80      0.75      0.77      1600
        weighted avg       0.82      0.83      0.83      1600



In [29]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

models = {
    "DecisionTree (depth6, bal)": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=42),
    "RandomForest":               RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                                                         n_jobs=-1, random_state=42),
    "GradientBoosting (HGB)":      HistGradientBoostingClassifier(max_depth=6, random_state=42),
}
for name, m in models.items():
    m.fit(X_train, y_train)
    p = m.predict(X_test)
    print(f"{name:28s} acc={accuracy_score(y_test, p):.3f}  macro-F1={f1_score(y_test, p, average='macro'):.3f}")

DecisionTree (depth6, bal)   acc=0.847  macro-F1=0.795
RandomForest                 acc=0.855  macro-F1=0.800
GradientBoosting (HGB)       acc=0.863  macro-F1=0.819
